# Session 13: Database Schema Design & Security

> Build scalable, normalized schemas with encryption at rest and audit trails.

## 1. Normalization & Scalable Design

**Normalization** is the process of organising a database schema to eliminate redundancy and ensure data integrity. Each Normal Form (NF) builds on the previous:

- **1NF:** Each column holds a single atomic value (no comma-separated lists, no arrays in a cell). Each row is uniquely identifiable.
- **2NF:** Every non-key column depends on the *whole* primary key, not just part of it. Eliminates partial dependencies. Relevant when using composite keys.
- **3NF:** Every non-key column depends *only* on the primary key — not on another non-key column (no transitive dependencies). Example: if a table stores `zip_code` and `city`, `city` depends on `zip_code`, not the primary key — move `city` to a separate `zip_codes` table.

**Why normalization matters:**
- **Update anomalies:** Without 3NF, changing a customer's city requires updating every row in the `orders` table that references them.
- **Insert anomalies:** You can't add a new city without inserting an order.
- **Delete anomalies:** Deleting the last order for a customer loses their address information.

---

**When to intentionally denormalize:**
Normalization is the correct starting point, but there are cases where controlled denormalization improves read performance:
- **Reporting / analytics:** A pre-aggregated `daily_sales_summary` table avoids joining 5 tables on every dashboard load.
- **High-read-volume tables:** Storing a denormalized `full_name` alongside `first_name` / `last_name` saves a string concatenation on millions of rows.

The key word is *intentional*: denormalize as a measured performance decision, with documentation, not as an accident of poor design.

---

**Scalable design principles:**
1. **Use surrogate keys (UUID or auto-increment)** rather than natural keys (email, SSN) as primary keys — natural keys can change, foreign key updates are painful.
2. **Index foreign keys** by default — joins without indexes produce full table scans.
3. **Avoid nullable foreign keys** where possible — a `NULL` product_id on an order row is usually a data integrity bug.
4. **Design for the access patterns** — understand how data will be queried before finalising the schema.

In [ ]:
# Schema evolution: 1NF → 3NF
bad_schema = 'orders(id, customer_name, customer_email, item1_name, item1_price, item2_name, item2_price)'

good_schema = '''
-- Normalized to 3NF
CREATE TABLE customers (
    id          UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    email       VARCHAR(255) UNIQUE NOT NULL,
    name        VARCHAR(255) NOT NULL,
    created_at  TIMESTAMPTZ DEFAULT NOW()
);
CREATE TABLE products (
    id          UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    name        VARCHAR(255) NOT NULL,
    price_cents INTEGER NOT NULL CHECK (price_cents > 0),
    category_id UUID REFERENCES categories(id)
);
CREATE TABLE orders (
    id          UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    customer_id UUID REFERENCES customers(id) ON DELETE RESTRICT,
    status      VARCHAR(50) NOT NULL DEFAULT 'pending',
    created_at  TIMESTAMPTZ DEFAULT NOW()
);
CREATE TABLE order_items (
    id          UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    order_id    UUID REFERENCES orders(id) ON DELETE CASCADE,
    product_id  UUID REFERENCES products(id),
    quantity    SMALLINT NOT NULL CHECK (quantity > 0),
    unit_price_cents INTEGER NOT NULL  -- snapshot at time of order
);
'''
print(good_schema)

## 💡 Additional Examples: Database Design

**Example 1 — The many-to-many pattern and junction tables**
A many-to-many relationship (e.g., students can enrol in many courses, and a course has many students) cannot be represented with two tables alone. You need a junction table (`enrollments`) with foreign keys to both sides. The junction table itself can carry data about the relationship (e.g., `enrolled_at`, `grade`). This is a fundamental normalisation pattern that new developers often try to work around by storing comma-separated IDs in a column — which breaks 1NF and makes every query painful.

**Example 2 — Soft delete pattern**
Hard-deleting rows (`DELETE FROM users WHERE id = ?`) is often the wrong choice for business data. If a customer deletes their account, you may still need their data for invoices, audit logs, or fraud analysis. The soft delete pattern adds a `deleted_at TIMESTAMP` column. Queries filter on `WHERE deleted_at IS NULL`. The row is still present for historical lookups. The trade-off is that every query must include the filter — forgetting it causes data leaks. A database view or ORM-level global filter can enforce this automatically.

**Example 3 — The audit log table pattern**
Regulated industries (finance, healthcare) require knowing who changed what data and when. Rather than storing `updated_by` and `updated_at` in every table, an audit log table (`audit_log`) captures every write operation: `table_name`, `row_id`, `changed_by`, `changed_at`, `old_value` (JSON), `new_value` (JSON). This separates the audit concern from the business data schema and provides a complete, queryable history. Implement it as a database trigger or an ORM event listener so it's impossible to forget.

In [ ]:
# ─── Example 1: Schema Migration with Version Control ────────────────────────
# Simulates how Alembic / Flyway manage DB migrations

migrations = [
    {
        'version': 'V001__create_users_table',
        'description': 'Initial users table',
        'up': '''
CREATE TABLE users (
    id         UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    email      VARCHAR(255) UNIQUE NOT NULL,
    created_at TIMESTAMPTZ DEFAULT NOW()
);
CREATE INDEX idx_users_email ON users(email);
        ''',
        'down': 'DROP TABLE IF EXISTS users;',
        'checksum': 'a1b2c3d4',
    },
    {
        'version': 'V002__add_user_profile',
        'description': 'Add profile fields to users',
        'up': '''
ALTER TABLE users ADD COLUMN full_name VARCHAR(255);
ALTER TABLE users ADD COLUMN phone_number VARCHAR(20);
ALTER TABLE users ADD COLUMN is_active BOOLEAN DEFAULT TRUE NOT NULL;
ALTER TABLE users ADD COLUMN last_login_at TIMESTAMPTZ;
        ''',
        'down': '''
ALTER TABLE users DROP COLUMN IF EXISTS full_name;
ALTER TABLE users DROP COLUMN IF EXISTS phone_number;
ALTER TABLE users DROP COLUMN IF EXISTS is_active;
ALTER TABLE users DROP COLUMN IF EXISTS last_login_at;
        ''',
        'checksum': 'e5f6g7h8',
    },
    {
        'version': 'V003__create_roles_and_permissions',
        'description': 'RBAC: roles and permissions tables',
        'up': '''
CREATE TABLE roles (
    id   UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    name VARCHAR(50) UNIQUE NOT NULL  -- 'admin', 'editor', 'viewer'
);
CREATE TABLE user_roles (
    user_id UUID REFERENCES users(id)  ON DELETE CASCADE,
    role_id UUID REFERENCES roles(id)  ON DELETE CASCADE,
    PRIMARY KEY (user_id, role_id)  -- Composite PK prevents duplicates
);
        ''',
        'down': '''
DROP TABLE IF EXISTS user_roles;
DROP TABLE IF EXISTS roles;
        ''',
        'checksum': 'i9j0k1l2',
    },
]

def simulate_migration_run(migrations, current_version=None):
    print('📦 Migration Engine\n' + '=' * 50)
    applied = []
    for m in migrations:
        if current_version and m['version'] <= current_version:
            print(f'  ⏭  SKIP   {m["version"]} (already applied)')
            continue
        print(f'  ✅ APPLY  {m["version"]}')
        print(f'         {m["description"]}')
        print(f'         Checksum: {m["checksum"]}')
        applied.append(m['version'])
    print(f'\n{len(applied)} migration(s) applied.')
    return applied

simulate_migration_run(migrations, current_version='V001__create_users_table')

# ─── Example 2: Soft Delete Pattern with Query Builder ───────────────────────
from datetime import datetime, timezone
from typing import Optional

class UserQueryBuilder:
    """
    Soft Delete pattern: records are never physically removed — only marked with
    a deleted_at timestamp. Every query must filter WHERE deleted_at IS NULL by default.
    """
    def __init__(self):
        self._filters = ['deleted_at IS NULL']  # ← Default safety filter — always included
        self._limit: Optional[int] = None
        self._order = 'created_at DESC'

    def active_only(self):
        self._filters.append('is_active = TRUE')
        return self

    def by_role(self, role: str):
        self._filters.append(f"role = '{role}'")
        return self

    def created_after(self, date_str: str):
        self._filters.append(f"created_at >= '{date_str}'")
        return self

    def limit(self, n: int):
        self._limit = n
        return self

    def build(self) -> str:
        where = ' AND '.join(self._filters)
        query = f'SELECT * FROM users WHERE {where} ORDER BY {self._order}'
        if self._limit:
            query += f' LIMIT {self._limit}'
        return query

print('\n=== Soft Delete Query Examples ===')

normal_query = UserQueryBuilder().build()
print(f'Default (excludes deleted):\n  {normal_query}\n')

filtered = UserQueryBuilder().active_only().by_role('admin').limit(10).build()
print(f'Active admins:\n  {filtered}\n')

recent = UserQueryBuilder().active_only().created_after('2026-01-01').build()
print(f'Active users since Jan 2026:\n  {recent}\n')

# Admin audit query that intentionally includes deleted records
admin_audit_query = (
    'SELECT *, deleted_at, deleted_by FROM users '
    "WHERE role = 'admin' "  # No deleted_at filter — intentional for audit
    'ORDER BY created_at DESC'
)
print(f'Admin audit (includes deleted):\n  {admin_audit_query}')

# ─── Example 3: Database Connection Pooling + Health Check ───────────────────
from dataclasses import dataclass, field
from typing import Optional
import random
import time

@dataclass
class ConnectionStats:
    pool_size: int
    checked_out: int
    idle: int
    overflow: int
    total_connections_created: int
    avg_checkout_time_ms: float

class MockConnectionPool:
    """
    Simulates a SQLAlchemy-style connection pool.
    Understanding pool mechanics helps debug connection leaks and timeout issues.
    """
    def __init__(self, pool_size: int = 10, max_overflow: int = 5):
        self.pool_size = pool_size
        self.max_overflow = max_overflow
        self._idle = list(range(pool_size))        # Available connections
        self._checked_out: set = set()
        self._overflow_count = 0
        self._total_created = pool_size
        self._checkout_times: list[float] = []

    def checkout(self) -> Optional[int]:
        if self._idle:
            conn = self._idle.pop()
            self._checked_out.add(conn)
            self._checkout_times.append(random.uniform(0.1, 2.5))
            return conn
        if self._overflow_count < self.max_overflow:
            # Create an overflow connection beyond the base pool size
            conn_id = self._total_created
            self._total_created += 1
            self._overflow_count += 1
            self._checked_out.add(conn_id)
            print(f'  ⚡ Overflow connection created (#{conn_id}), overflow={self._overflow_count}')
            return conn_id
        return None  # Pool exhausted — caller must handle this

    def checkin(self, conn_id: int):
        self._checked_out.discard(conn_id)
        if conn_id >= self.pool_size:  # Overflow connection — discard, do not recycle
            self._overflow_count -= 1
        else:
            self._idle.append(conn_id)

    def stats(self) -> ConnectionStats:
        avg = sum(self._checkout_times) / len(self._checkout_times) if self._checkout_times else 0
        return ConnectionStats(
            pool_size=self.pool_size,
            checked_out=len(self._checked_out),
            idle=len(self._idle),
            overflow=self._overflow_count,
            total_connections_created=self._total_created,
            avg_checkout_time_ms=round(avg, 2),
        )

pool = MockConnectionPool(pool_size=5, max_overflow=3)
print('\n=== Connection Pool Simulation ===')

# Simulate concurrent checkouts that exceed the pool size → triggers overflow
connections = [pool.checkout() for _ in range(7)]
stats = pool.stats()
print(f'After 7 checkouts: checked_out={stats.checked_out}, idle={stats.idle}, overflow={stats.overflow}')

# Return some connections back to the pool
for c in connections[:3]: pool.checkin(c)
stats = pool.stats()
print(f'After 3 returns:   checked_out={stats.checked_out}, idle={stats.idle}, overflow={stats.overflow}')

# Demonstrate pool exhaustion
extra_conns = [pool.checkout() for _ in range(5)]
exhausted = pool.checkout()
print(f'Pool exhausted: checkout() returned {exhausted}  ← None means no connection available!')
print(f'Final stats: {pool.stats()}')


## 2. Data Encryption & Audit Trails

**Encryption at Rest:**
```sql
-- Store sensitive fields encrypted
CREATE TABLE payment_methods (
    id           UUID PRIMARY KEY,
    user_id      UUID REFERENCES users(id),
    card_last4   CHAR(4),           -- only store last 4
    card_token   TEXT,              -- tokenized by payment provider
    -- NEVER store: full card number, CVV, PIN
);
```

**Audit Trail Pattern:**
```sql
CREATE TABLE audit_log (
    id          UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    table_name  VARCHAR(100),
    record_id   UUID,
    action      VARCHAR(10),  -- INSERT/UPDATE/DELETE
    changed_by  UUID REFERENCES users(id),
    old_values  JSONB,
    new_values  JSONB,
    changed_at  TIMESTAMPTZ DEFAULT NOW()
);
```

## 3. Practice

### 🧪 AI Lab / Practice

> **TODO:** Design a complete schema for a module from your project: (1) ERD in Mermaid with all relationships, (2) DDL SQL with constraints and indexes, (3) Identify which fields need encryption and why, (4) Add audit trail for sensitive tables. Review with team before next session.

In [ ]:
# TODO: Implement your solution here
raise NotImplementedError('Not implemented yet — complete this exercise')